04_models_arima_sarimax

**BLOQUE 2 — Librerías y configuración**

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.graphics.tsaplots import plot_acf

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

**BLOQUE 3 — Rutas**

In [ ]:
# ==========================================
# BLOQUE 3 — Configuración de rutas
# ==========================================

from pathlib import Path

# Detectar si está dentro de notebooks/
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
OUTPUTS_FIGURES = PROJECT_ROOT / "outputs" / "figuras"
OUTPUTS_TABLES = PROJECT_ROOT / "outputs" / "tablas"

# Crear carpetas si no existen
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
OUTPUTS_FIGURES.mkdir(parents=True, exist_ok=True)
OUTPUTS_TABLES.mkdir(parents=True, exist_ok=True)

# Archivo de entrada
INPUT_FILE = DATA_PROCESSED / "data_daily_ready.csv"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Archivo de entrada:", INPUT_FILE)

# Validación
if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"No se encontró el archivo: {INPUT_FILE}\n"
        "Verifica que el archivo exista en data/processed/"
    )

**BLOQUE 4 — Carga de datos**

In [ ]:
df = pd.read_csv(INPUT_FILE, parse_dates=["date"])

DATE_COL = "date"
Y_COL = "target_kwh"

df = df.sort_values(DATE_COL).set_index(DATE_COL)

print("Dimensión del dataset:", df.shape)
print("Frecuencia inferida:", pd.infer_freq(df.index))

display(df.head())

**BLOQUE 5 — División temporal train/test**

In [ ]:
train = df.loc["2022-01-01":"2024-12-31"].copy()
test = df.loc["2025-01-01":"2025-12-10"].copy()

y_train = train[Y_COL]
y_test = test[Y_COL]

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Rango train:", train.index.min(), "->", train.index.max())
print("Rango test :", test.index.min(), "->", test.index.max())

**BLOQUE 6 — Funciones de métricas**

In [ ]:
def mape(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

def compute_metrics(y_true, y_pred, model_name):
    return {
        "model": model_name,
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAPE": mape(y_true, y_pred)
    }

**BLOQUE 7 — Ajuste de candidatos ARIMA**

In [ ]:
arima_candidates = {
    "ARIMA(0,1,1)": (0, 1, 1),
    "ARIMA(1,1,0)": (1, 1, 0),
    "ARIMA(1,1,1)": (1, 1, 1),
}

arima_results = {}
arima_summary_rows = []

for name, order in arima_candidates.items():
    model = ARIMA(
        y_train,
        order=order,
        enforce_stationarity=False,
        enforce_invertibility=False
    )
    fit = model.fit()
    arima_results[name] = fit

    arima_summary_rows.append({
        "model": name,
        "order": str(order),
        "AIC": fit.aic,
        "BIC": fit.bic,
        "HQIC": fit.hqic,
        "logLik": fit.llf
    })

arima_comparison = pd.DataFrame(arima_summary_rows).sort_values("AIC").reset_index(drop=True)

display(arima_comparison)
arima_comparison.to_csv(OUTPUTS_TABLES / "table_arima_candidate_comparison.csv", index=False)

**BLOQUE 8 — Selección del mejor ARIMA**

In [ ]:
best_arima_name = arima_comparison.iloc[0]["model"]
best_arima_fit = arima_results[best_arima_name]

print("Mejor modelo ARIMA según AIC:", best_arima_name)
print(best_arima_fit.summary())

**BLOQUE 9 — Diagnóstico de residuos ARIMA**

In [ ]:
arima_resid = best_arima_fit.resid.dropna()

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

axes[0].plot(arima_resid.index, arima_resid, linewidth=0.8)
axes[0].axhline(0, color="black", linestyle="--", linewidth=1)
axes[0].set_title(f"Residuos del modelo {best_arima_name}")
axes[0].set_xlabel("Fecha")
axes[0].set_ylabel("Residuo")
axes[0].grid(True, alpha=0.3)

plot_acf(arima_resid, lags=40, ax=axes[1], alpha=0.05)
axes[1].set_title(f"ACF de residuos - {best_arima_name}")

plt.tight_layout()
plt.savefig(OUTPUTS_FIGURES / "diagnostico_residuos_best_arima.png", dpi=300, bbox_inches="tight")
plt.show()

**BLOQUE 10 — Predicción ARIMA en test**

In [ ]:
arima_forecast = best_arima_fit.forecast(steps=len(y_test))
arima_forecast.index = y_test.index

arima_metrics = compute_metrics(y_test, arima_forecast, best_arima_name)
display(pd.DataFrame([arima_metrics]))

arima_pred_df = pd.DataFrame({
    "date": y_test.index,
    "y_true": y_test.values,
    "y_pred_arima": arima_forecast.values
})

arima_pred_df.to_csv(OUTPUTS_TABLES / "table_arima_test_predictions.csv", index=False)

**BLOQUE 11 — Gráfico real vs predicción ARIMA**

In [ ]:
fig, ax = plt.subplots(figsize=(15, 4.8))

ax.plot(y_test.index, y_test.values, label="Real", linewidth=1.2)
ax.plot(arima_forecast.index, arima_forecast.values, label=f"Predicción {best_arima_name}", linewidth=1.2)

ax.set_title(f"Predicción en test - {best_arima_name}", fontsize=13)
ax.set_xlabel("Fecha")
ax.set_ylabel("Demanda eléctrica diaria (kWh)")
ax.grid(True, alpha=0.3)
ax.legend(frameon=False)

ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))

plt.tight_layout()
plt.savefig(OUTPUTS_FIGURES / "forecast_test_best_arima.png", dpi=300, bbox_inches="tight")
plt.show()

**BLOQUE 12 — Definición de variables exógenas para SARIMAX**

In [ ]:
exog_cols = [
    "t_mean_c",
    "t_max_c",
    "t_min_c",
    "rh_mean_pct",
    "precip_mm",
    "sun_hours",
    "sst_c",
    "dow",
    "is_weekend",
    "is_business_day",
    "is_holiday",
    "is_pre_holiday",
    "is_post_holiday",
    "is_long_weekend"
]

missing_exog = [col for col in exog_cols if col not in df.columns]
if missing_exog:
    raise ValueError(f"Faltan columnas exógenas: {missing_exog}")

X_train = train[exog_cols].copy()
X_test = test[exog_cols].copy()

print("Exógenas train:", X_train.shape)
print("Exógenas test :", X_test.shape)
display(X_train.head())

**BLOQUE 13 — Ajuste del modelo SARIMAX**

In [ ]:
best_order = eval(arima_comparison.iloc[0]["order"])

sarimax_model = SARIMAX(
    y_train,
    exog=X_train,
    order=best_order,
    enforce_stationarity=False,
    enforce_invertibility=False
)

sarimax_fit = sarimax_model.fit(disp=False)

print("Modelo SARIMAX ajustado con order =", best_order)
print(sarimax_fit.summary())

**BLOQUE 14 — Diagnóstico de residuos SARIMAX**

In [ ]:
sarimax_resid = sarimax_fit.resid.dropna()

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

axes[0].plot(sarimax_resid.index, sarimax_resid, linewidth=0.8)
axes[0].axhline(0, color="black", linestyle="--", linewidth=1)
axes[0].set_title("Residuos del modelo SARIMAX")
axes[0].set_xlabel("Fecha")
axes[0].set_ylabel("Residuo")
axes[0].grid(True, alpha=0.3)

plot_acf(sarimax_resid, lags=40, ax=axes[1], alpha=0.05)
axes[1].set_title("ACF de residuos - SARIMAX")

plt.tight_layout()
plt.savefig(OUTPUTS_FIGURES / "diagnostico_residuos_sarimax.png", dpi=300, bbox_inches="tight")
plt.show()

**BLOQUE 15 — Predicción SARIMAX en test**

In [ ]:
sarimax_forecast = sarimax_fit.forecast(steps=len(y_test), exog=X_test)
sarimax_forecast.index = y_test.index

sarimax_metrics = compute_metrics(y_test, sarimax_forecast, f"SARIMAX{best_order}")
display(pd.DataFrame([sarimax_metrics]))

sarimax_pred_df = pd.DataFrame({
    "date": y_test.index,
    "y_true": y_test.values,
    "y_pred_sarimax": sarimax_forecast.values
})

sarimax_pred_df.to_csv(OUTPUTS_TABLES / "table_sarimax_test_predictions.csv", index=False)

**BLOQUE 16 — Gráfico real vs predicción SARIMAX**

In [ ]:
fig, ax = plt.subplots(figsize=(15, 4.8))

ax.plot(y_test.index, y_test.values, label="Real", linewidth=1.2)
ax.plot(sarimax_forecast.index, sarimax_forecast.values, label=f"Predicción SARIMAX{best_order}", linewidth=1.2)

ax.set_title(f"Predicción en test - SARIMAX{best_order}", fontsize=13)
ax.set_xlabel("Fecha")
ax.set_ylabel("Demanda eléctrica diaria (kWh)")
ax.grid(True, alpha=0.3)
ax.legend(frameon=False)

ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))

plt.tight_layout()
plt.savefig(OUTPUTS_FIGURES / "forecast_test_sarimax.png", dpi=300, bbox_inches="tight")
plt.show()

**BLOQUE 17 — Comparación ARIMA vs SARIMAX**

In [ ]:
comparison_metrics = pd.DataFrame([arima_metrics, sarimax_metrics]).sort_values("RMSE").reset_index(drop=True)
display(comparison_metrics)

comparison_metrics.to_csv(OUTPUTS_TABLES / "table_arima_vs_sarimax_metrics.csv", index=False)

**BLOQUE 18 — Tabla de hiperparámetros**

In [ ]:
hyperparams_table = pd.DataFrame({
    "model": [
        best_arima_name,
        f"SARIMAX{best_order}"
    ],
    "order": [
        str(best_order),
        str(best_order)
    ],
    "seasonal_order": [
        "None",
        "None"
    ],
    "exogenous_variables": [
        "No",
        ", ".join(exog_cols)
    ],
    "enforce_stationarity": [
        False,
        False
    ],
    "enforce_invertibility": [
        False,
        False
    ]
})

display(hyperparams_table)
hyperparams_table.to_csv(OUTPUTS_TABLES / "table_hyperparameters_arima_sarimax.csv", index=False)